# 11 — Validação Cruzada, Seleção e Otimização

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/11_validacao_otimizacao.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_validacao_otimizacao.pdf)
- [Documentação da área](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Projetar busca de hiperparâmetros com pipelines, entender GridSearchCV, RandomizedSearchCV, validação estratificada/por grupos/temporal e noções de validação aninhada.



## 2. Fundamentos matemáticos

Para $K$ folds, $\mathrm{CV}(\theta)=\frac1K\sum_{k=1}^K\widehat R_k(\theta)$. A escolha $\hat\theta=\arg\min_\theta\mathrm{CV}(\theta)$ depende dos mesmos folds avaliados; para estimar sem esse viés de seleção, usar um teste independente (ou validação aninhada).

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
X,y=load_wine(return_X_y=True)
a,b,c,d=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
pipe=make_pipeline(StandardScaler(),KNeighborsClassifier())
grid=GridSearchCV(pipe,{'kneighborsclassifier__n_neighbors':[3,5,7,9],'kneighborsclassifier__weights':['uniform','distance']},cv=StratifiedKFold(4,shuffle=True,random_state=42),scoring='balanced_accuracy',n_jobs=1)
grid.fit(a,c)
print('Melhores parâmetros:',grid.best_params_,'CV:',round(grid.best_score_,3))
print('Teste final (uma vez):',round(balanced_accuracy_score(d,grid.predict(b)),3))

Melhores parâmetros: {'kneighborsclassifier__n_neighbors': 7, 'kneighborsclassifier__weights': 'uniform'} CV: 0.969
Teste final (uma vez): 0.981


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
r=RandomizedSearchCV(make_pipeline(StandardScaler(),KNeighborsClassifier()),{'kneighborsclassifier__n_neighbors':randint(1,19)},n_iter=6,cv=4,random_state=42,n_jobs=1)
r.fit(a,c)
print('RandomizedSearchCV melhor k:',r.best_params_['kneighborsclassifier__n_neighbors'])

RandomizedSearchCV melhor k: 11


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: quantos ajustes executa uma busca 4x3 com 5 folds?
params_1,params_2,folds=4,3,5
print('Ajustes de validação:',params_1*params_2*folds,'+ refit final no treino')

Ajustes de validação: 60 + refit final no treino


## 6. Interpretação, limitações e erros comuns

A busca exaustiva pode ficar cara em espaços grandes. `n_jobs=1` aqui evita excesso de memória no computador de 8 GB. Em dados temporais ou agrupados, substituir os folds padrão por esquemas adequados.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- `GridSearchCV`: grade fixa; `RandomizedSearchCV`: amostragem de combinações.
- `scoring` deve representar a decisão de negócio.
- Teste é avaliação final, não um instrumento de tuning.
- Registrar `random_state`, versões e tempo de execução.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 08-cross-validation.pdf; 09-grid-search.pdf.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** seleção aninhada. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **seleção aninhada** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Validação cruzada estratificada

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.


In [4]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
y = np.array([0]*12 + [1]*6)
for treino, teste in StratifiedKFold(3, shuffle=True, random_state=42).split(np.zeros(len(y)), y):
    print('Treino', np.bincount(y[treino]), 'Teste', np.bincount(y[teste]))

Treino [8 4] Teste [4 2]
Treino [8 4] Teste [4 2]
Treino [8 4] Teste [4 2]


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **seleção aninhada** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
